Install the following python packages using your package manager of choice (pip in this case)

In [2]:
!pip install numpy==2.2
!pip install matplotlib
!pip install scipy
!pip install pandas
!pip install f90nml
!pip install numba
!pip install plotly
!pip install bokeh
!pip install astropy
!pip install ipyformkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 11.2 MB/s eta 0:00:00 MB/s eta 0:00:01:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6


Import the `m3dis` python package amongst other common packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import m3dis
plt.style.use('m3dis/Bergemann2020.mplstyle')
#plt.style.use('m3dis/Bergemann2023.mplstyle') # uncomment for serif axis labels
%config InlineBackend.figure_format='svg' # turns plots into vector graphics inside this notebook

Execute Multi by calling the Dispatch executable and supplying the default demo input parameter file `demo.nml`.
This will run a 1D solar Marcs model using a 6-Level toy Barium atom.

In [ ]:
! ./dispatch.x demo.nml

The default output directory is `./data/`. A new subfolder is created according to the name of your input parameter file. It can be opened and read as follows:

In [27]:
run = m3dis.read('data/demo/')

Now we have created an object with the name of `run`. This object contains all of the data and meta-data of our demo run. The spectral data is contained in run.lam and run.flux. One can easily plot 
the spectrum using the `plot_spectrum()` function (which calls `plt.plot()` and all of its arguments):

In [ ]:
plt.plot(run.lam, run.flux/run.flux_cnt) # or alternatively
lam, flux, plot = run.plot_spectrum(linestyle=':')

To zoom into a part of the spectrum one may do:

In [ ]:
lam0 = 4554.033
dlam = 2.5
lam, flux, plot = run.plot_spectrum(lam0=lam0, dlam=dlam)

`plot_spectrum()` trims the underlying spectral data using the  `crop()` function of the `run` object. The `crop()` function only returns the wavelengths and flux/intensity without plotting it. It has many useful arguments, which are also accepted by `plot_spectrum()`. Here I list the most important ones:
1. `lam0`, `dlam`: can be used in combination in order to crop the spectrum to a region around a central wavelength lam0. dlam may also be a tuple, e.g. (-0.5, 0.2), which would result in asymmetric cropping aroun lam0.
2. `xmin`,`xmax`: alternatively one may pass xmin and xmax instead to crop the wavelengths.
3. `lam_offset=False`: when one uses lam0 and dlam, the wavelength values are given as distance from lam0, but by setting lam_offset=False, one can keep the original wavelengths.
4. `norm=False`: plots the unnormalised spectrum.
5. `imu=0`: plots the disk-center intensity, rather than the flux. By increasing imu, you can plot the various angles of the chosen angle quadrature of the run.
6. `vdop=1.5`: applies doppler broadening with a velocity of 1.5 km/s by convolving with a Gaussian profile.
7. `vshift=1.5`: doppler shifts the spectrum corresponding to a velocity of 1.5 km/s.
8. `LTE=True`: plots the LTE spectrum. By default it will try to plot the NLTE spectrum.


An array accessible via `run.line` contains multiple objects of the `trans_class`. This class makes plotting of spectral lines stored in the model atom very easy. Each run stores both LTE and NLTE spectra by default. Let's make a quick comparison:

In [ ]:
imu = None # None gives flux, use integers to plot intensities for different angles
norm = True # set to False for unnormalised line profile
ka = 0 # atom index
kr = 0 # line index

line = run.atoms[ka].line[kr]
line.plot(imu=imu, norm=norm, LTE=False, linestyle='-', label='non-LTE')
line.plot(imu=imu, norm=norm, LTE=True, linestyle='--', label='LTE')
plt.legend()

In [ ]:
dlam = 0.8 # zooms into the line profile to ±dlam angstroms.
norm = False

# alternatively one may get the line object by its wavelength
line = run.get_line(4554.033)

 # iterate over all angles
for imu, mu in enumerate(run.mus):
  label = r'$\mu=${:.2f}'.format(mu)
  line.plot(imu=imu, norm=norm, dlam=dlam, label=label)

plt.ylabel('Intensity [erg/s/cm$^2$/sr/Hz]')
plt.legend()

In [ ]:
dlam = 0.5
imu = 0
line = run.get_line(4554.033)

 # convolve with doppler profiles
for vdop in [1.0, 3.0, 5.0]:
  label = 'v={:.1f} km/s'.format(vdop)
  line.plot(imu=imu, dlam=dlam, vdop=vdop, label=label)

plt.legend()

Here is a quick example of how to save the spectral line profile as an ascii table. In this case we decide to save the Flux as well as the individual mu-angles.

In [ ]:
line = run.get_line(4554.033)
line.prepare()
data = {}
data['Wavelength'] = line.lam
data['Flux'] = line.flux / line.flux_cnt

# save angle resolved line profiles as ascii table
for imu, mu in enumerate(run.mus):
  label = '{:.3f}'.format(mu)
  data[label] = line.ie[:,imu] / line.cnt[:,imu]

df = pd.DataFrame(data=data)
ascii_table = df.to_string(index=False)
print(ascii_table)

Let's download a 3D solar atmosphere snapshot from our website at nlte.mpia.de.

In [1]:
from urllib import request
import zipfile
import os

# Define the URL and output paths
url = 'https://gemini-web.mpia.de/atmos3D/SunLTE_E_t57.77g44.40m0.000_v1.0/SunLTE_t5777_g4.440_z0.000_sn779_m3dis.zip'
out_dir = 'input_multi3d/atmos/'
tmp = out_dir + 'SunLTE_m3dis.zip'

# Download the file
print("Downloading...")
request.urlretrieve(url, tmp)
print("Download complete.")

# Unzip the file
print("Unzipping...")
with zipfile.ZipFile(tmp, 'r') as zip_ref:
    zip_ref.extractall(out_dir)
print("Unzipping complete.")

# remove the zip file
os.remove(tmp)

Downloading...
Download complete.
Unzipping...
Unzipping complete.


Now we can run the 3D example. This may take a while, depending on your machine. The output will be stored in data/demo3d/.

In [ ]:
! ./dispatch.x demo3d.nml